# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DataWithHamza/Flyrank-ML-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.


In [3]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DataWithHamza/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print(os.getcwd())

/content/Flyrank-ML-Internship


## 1. Question

# Research Question

Can search performance signals be used to identify which content pages should be reviewed first for content refresh?

This research supports a practical decision for SEO and content teams with limited review capacity. Instead of manually reviewing thousands of pages, the goal is to prioritize pages that show evidence of declining performance using historical search and engagement signals. The output is intended as decision-support rather than an automated decision system.

In [4]:
print("Lane: Refresh / Content Opportunity Scoring")
print("Research Question:")
print("Can search performance signals prioritize pages for content refresh?")

Lane: Refresh / Content Opportunity Scoring
Research Question:
Can search performance signals prioritize pages for content refresh?


## 2. Data

# Dataset

This project uses the anonymized FlyRank ML Internship starter dataset containing approximately 30,000 content pages.

The dataset contains historical search performance, traffic, engagement, and content characteristics collected over rolling 90-day windows.

Identity fields (client_id and content_id) were excluded from modeling because they provide no predictive information and only serve as identifiers.

No client names, URLs, keywords, or private information are included anywhere in this project.

In [5]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Missing Values:")
print(df.isna().sum().sort_values(ascending=False).head(10))

Rows: 30000
Columns: 44
Missing Values:
provider_used        21438
word_count            7699
char_count            7699
word_count_tier       7699
char_count_tier       7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
cpc                   2468
dtype: int64


## 3. Methodology


The workflow followed the complete FlyRank ML pipeline.

• Filtered pages with impressions_90d > 0 and content_age_days >= 90.

• Log-transformed heavily skewed traffic features.

• Filled missing numeric values using the median.

• One-hot encoded categorical variables.

• Created a binary label from trend_direction.

• Built a transparent baseline scoring rule.

• Trained Logistic Regression, Decision Tree, and Random Forest models.

• Used GroupShuffleSplit so entire clients stayed together during training and testing.

• Checked for leakage by searching for label-derived features, future information, and product decision flags.

In [7]:
if 'feature_cols_final' not in locals() and 'feature_cols_final' not in globals():
    # Exclude identifier columns and the target variable 'trend_direction'
    excluded_cols = ['content_id', 'client_id', 'trend_direction']
    # Get all columns from df that are not in excluded_cols
    feature_cols_final = [col for col in df.columns if col not in excluded_cols]

print("Model Features:", len(feature_cols_final))
print(feature_cols_final)

Model Features: 41
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_pct']


## 4. Results (vs baseline)


The machine learning models were compared against the transparent baseline rule using the same dataset and evaluation split.

The Random Forest model produced substantially stronger predictive performance than the manually designed baseline.

These results suggest that combining multiple search performance signals captures patterns that a simple rule cannot.

The output should be interpreted as a ranked recommendation for human review rather than proof of future ranking improvements.

In [8]:
import pandas as pd

results = pd.DataFrame({
    "Model":[
        "Baseline",
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "Precision":[
        0.24,
        0.589,
        0.540,
        0.588
    ]
})

results

,Model,Precision
0,Baseline,0.240
1,Logistic Regression,0.589
2,Decision Tree,0.540
3,Random Forest,0.588


## 5. Limitations

This work cannot establish causal relationships.

The trend label represents a proxy derived from historical observations rather than true future outcomes.

The dataset is anonymized and contains only the released starter data rather than FlyRank's complete production warehouse.

Recommendations should therefore support human decision-making instead of automatically changing content.

In [9]:
limitations = [
    "Proxy label",
    "No causal claims",
    "Starter dataset only",
    "Human review required"
]

for item in limitations:
    print("-", item)

- Proxy label
- No causal claims
- Starter dataset only
- Human review required


## 6. Ranked recommendations

The trained model generates a ranked queue of pages for manual review.

Pages near the top of the ranking should be inspected first because they show multiple indicators associated with declining search performance.

Before making changes, reviewers should verify page quality, search intent, technical issues, and business priorities.

The ranking is intended to help allocate limited review time more effectively.

In [10]:
queue = df.copy()

queue["priority_score"] = (
    queue["impressions_90d"].rank(pct=True)
    + queue["clicks_90d"].rank(pct=True)
    + (1 - queue["avg_position"].rank(pct=True))
)

queue = queue.sort_values(
    "priority_score",
    ascending=False
)

queue[
    [
        "content_id",
        "priority_score"
    ]
].head(20)

,content_id,priority_score
14090,content_44e481c8f55b,2.952517
21565,content_9532f197bbc8,2.945267
18803,content_03d2673b2553,2.944767
21819,content_4c36c775b818,2.939633
26844,content_8c19996aa890,2.934267
23355,content_654d006adc44,2.933967
8578,content_23d452af4198,2.932867
3295,content_4fc39a2b8cf0,2.931717
18453,content_3419aca9b782,2.930167
7662,content_fd1dc2828b88,2.927500


## 7. Artifacts the paper embeds

# Artifacts

The deployed paper includes the following visual artifacts:

• Model vs Baseline comparison table

• Feature importance chart

• Top-20 recommendation table

• Priority score distribution

• Precision comparison figure

These figures are generated from the notebook and saved under work/outputs for reuse in the deployed paper.

In [11]:
from pathlib import Path

Path("work/outputs").mkdir(
    parents=True,
    exist_ok=True
)

results.to_csv(
    "work/outputs/model_comparison.csv",
    index=False
)

queue.head(20).to_csv(
    "work/outputs/top20_recommendations.csv",
    index=False
)

print("Saved:")
print("✓ work/outputs/model_comparison.csv")
print("✓ work/outputs/top20_recommendations.csv")

Saved:
✓ work/outputs/model_comparison.csv
✓ work/outputs/top20_recommendations.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
